# Agentic Portfolio Construction — The Reference Discovery Call

`pipeline_demo.ipynb` runs the pipeline on **BLS personas**: nine occupations, salaries
from the OES flat file, every field present by construction. This notebook runs it on a
**real conversation** — the discovery call with Priya Raman, in advisor prose notes, with
the gaps, hedges and contradictions a real call actually contains.

The difference is not cosmetic. A BLS persona arrives as a dict; a conversation arrives as
prose that has to be read, sourced, classified and *refused* where it says nothing. Three
things only exist on this path:

| | BLS persona | This transcript |
|---|---|---|
| Where fields come from | a table | a language model, with a quote per field |
| Fields never discussed | cannot happen | returned `UNKNOWN` + a follow-up question |
| `client_statements` | empty | classified, routed, and read by Compliance Job 3.2 |

That last row is why this notebook matters for the compliance layer. Job 3.2 checks the
proposed portfolio against the client's own hard constraints. For a BLS persona there are
no statements, so it passes **vacuously** — which in a report looks identical to a genuine
pass. This is the only run where it has something to check.

The transcript closes with its own instruction:

> *TODO: build her profile, run the human-capital decomposition, and SHOW her how much
> biotech risk she's carrying before she even opens a brokerage account.*

That is what §6 onwards does.

---

**Requires** `intake_priya_raman.txt` at the repo root, `ANTHROPIC_API_KEY` for the
structured extractor (§3), and the parquet cache in `data/storage/` for §7 onwards.
§3 caches its extraction to disk, so only the first run needs the key.

## 1 · Environment setup

Same helpers as `pipeline_demo.ipynb`. Agent `INFO` logs are routed into the notebook —
the feedback loops in §8 are only legible with them on.

In [1]:
import os, sys, json, logging
from pathlib import Path

import pandas as pd

PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

try:
    from dotenv import load_dotenv
    load_dotenv(Path(PROJECT_ROOT) / ".env")
    print(".env loaded")
except ImportError:
    print("dotenv not installed — run: pip install python-dotenv")

FRED_KEY      = os.environ.get("FRED_API_KEY")
ANTHROPIC_KEY = os.environ.get("ANTHROPIC_API_KEY")

print(f"FRED_API_KEY set:      {bool(FRED_KEY)}")
print(f"ANTHROPIC_API_KEY set: {bool(ANTHROPIC_KEY)}"
      f"{'' if ANTHROPIC_KEY else '   -> §3 falls back to the cached extraction if one exists'}")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)-34s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
    force=True,
)
log = logging.getLogger("priya-demo")

RULE = "=" * 88


def banner(text):
    print("\n" + "#" * 88 + "\n#  " + text + "\n" + "#" * 88)


def section(title):
    print("\n" + RULE + "\n" + title + "\n" + RULE)


def show(title, model):
    """Print a Pydantic model (or dict) as indented JSON under a titled rule."""
    section(title)
    if hasattr(model, "model_dump_json"):
        print(model.model_dump_json(indent=2))
    else:
        print(json.dumps(model, indent=2, default=str))


print("\nsetup complete — Python", sys.version.split()[0])

.env loaded
FRED_API_KEY set:      True
ANTHROPIC_API_KEY set: True

setup complete — Python 3.11.15


## 2 · The call

`intake_priya_raman.txt` is the reference discovery call. Two properties of it are worth
noticing before any extraction runs, because both broke the intake layer the first time:

1. **It is advisor prose notes, not a dialogue.** There are no `CLIENT:` lines. The
   rule-based extractor read only those, so it recovered *nothing* — and reported no error.
   Attribution in prose notes has to be decided per fact, from advisor-judgement cues
   ("I'd put her at…", "when I floated…"), not per line.
2. **It states employer stock as a dollar figure**, for a field the contract bounds to
   `[0, 1]`. The prompt never stated the unit, so the extractor returned `360000` for
   `RSU_concentration` and the failure surfaced at the bridge, one layer from its cause.

Neither was visible on the synthetic corpus, which is the argument for keeping a real
transcript in the test set.

In [2]:
from agents.profile import reference_case

TRANSCRIPT_PATH = reference_case.TRANSCRIPT
CLIENT_ID       = reference_case.CLIENT_ID

if not TRANSCRIPT_PATH.exists():
    raise FileNotFoundError(
        f"{TRANSCRIPT_PATH} not found.\n"
        "This notebook has nothing to run without the reference discovery call. "
        "Place intake_priya_raman.txt at the repo root and re-run this cell."
    )

transcript = reference_case.load_transcript()
lines      = transcript.splitlines()

section("THE TRANSCRIPT")
print(f"  path              {TRANSCRIPT_PATH.name}")
print(f"  client_id         {CLIENT_ID}")
print(f"  lines             {len(lines)}")
print(f"  characters        {len(transcript):,}")

# The format check that the rule-based extractor silently failed.
from agents.profile.intake import _is_turn_formatted

turn_formatted = _is_turn_formatted(lines)
print(f"  speaker turns     {turn_formatted}"
      f"{'' if turn_formatted else '   <-- prose notes: no CLIENT:/ADVISOR: lines to read'}")

section("FIRST 25 LINES")
for i, line in enumerate(lines[:25]):
    print(f"  {i:>3} | {line}")
print(f"\n  ... {max(0, len(lines) - 25)} more lines")


THE TRANSCRIPT
  path              intake_priya_raman.txt
  client_id         priya_raman
  lines             49
  characters        2,890
  speaker turns     False   <-- prose notes: no CLIENT:/ADVISOR: lines to read

FIRST 25 LINES
    0 | DISCOVERY CALL â€” intake notes (raw, unedited)
    1 | Advisor: J. Okafor   |   Prospective client: Priya Raman   |   45 min, phone
    2 | 
    3 | Priya reached out through the Friday referral. Below is roughly how the
    4 | conversation went â€” dumping it here before I forget the details, will clean up
    5 | into the system later.
    6 | 
    7 | She's 47, lives in Boston. Works as VP of Clinical Development at a mid-cap
    8 | biotech â€” the company is publicly traded (ticker ARVX). She's been there almost
    9 | nine years, came over from a big pharma. Says she "thinks like a scientist, not
   10 | a markets person," so she wants someone else handling the portfolio.
   11 | 
   12 | On comp: base salary is $310,000. On top of that t

## 3 · Extraction — the floor, then the structured extractor

Two extractors behind one protocol, run on the same text.

`RuleBasedExtractor` is deterministic, offline, and free. It is the floor an LLM has to
beat before its cost is justified — on the *synthetic* corpus it ties the structured
extractor at 98.6% field accuracy, so on that corpus the model buys nothing. This is the
transcript where that stops being true.

`StructuredExtractor` asks field by field, and each answer must carry a source, a
confidence, and a verbatim quote — or declare itself `UNKNOWN` and propose the question to
ask. The provenance is not advice to the model; §4 shows it is enforced by the contract.

In [3]:
from agents.profile.intake import RuleBasedExtractor

floor = RuleBasedExtractor().extract(transcript, CLIENT_ID, "priya_raman")
floor_got = sum(1 for f in floor.fields.values() if f.value is not None)

section("FLOOR — RuleBasedExtractor (deterministic, offline)")
print(f"  fields recovered      {floor_got} of {len(floor.fields)}")
print(f"  statements classified {len(floor.statements)}   (cannot classify — no taxonomy)")
print(f"\n  The floor is not bad code. It is a measurement: these are prose notes,")
print(f"  there is no CLIENT: line to key off, and the honest report of that is")
print(f"  {floor_got} of {len(floor.fields)} — reported, not silently returned as an empty profile.")


FLOOR — RuleBasedExtractor (deterministic, offline)
  fields recovered      1 of 15
  statements classified 0   (cannot classify — no taxonomy)

  The floor is not bad code. It is a measurement: these are prose notes,
  there is no CLIENT: line to key off, and the honest report of that is
  1 of 15 — reported, not silently returned as an empty profile.


In [4]:
from contracts import ExtractedProfile
from agents.profile.intake import StructuredExtractor

# Cache the extraction. The transcript never changes, and re-running a paid
# extraction on every notebook execution is the kind of cost nobody notices
# until the bill. Delete the file to force a fresh call.
CACHE = Path(PROJECT_ROOT) / "data" / "outputs" / "reference_case_priya.json"

def _load_cache():
    return ExtractedProfile.model_validate_json(CACHE.read_text(encoding="utf-8"))


if ANTHROPIC_KEY:
    try:
        extracted = StructuredExtractor().extract(transcript, CLIENT_ID, "priya_raman")
        CACHE.parent.mkdir(parents=True, exist_ok=True)
        CACHE.write_text(extracted.model_dump_json(indent=2), encoding="utf-8")
        print(f"extracted live, cached -> {CACHE.name}")
    except Exception as e:
        # Diagnose before retrying. A key that is present but unusable (no
        # credit, revoked, wrong org) fails identically to a bad prompt from
        # the caller's side, and re-running the extraction is the one response
        # that cannot help. Same lesson as the allocation agent's max_tokens
        # check: name the cause, do not send the loop back around.
        if not CACHE.exists():
            raise RuntimeError(
                f"Extraction failed and no cached copy exists at {CACHE.name}.\n"
                f"  cause: {type(e).__name__}: {e}\n"
                "If this is a billing or auth error the key is present but unusable — "
                "fix the account, do not re-run the cell."
            ) from e
        print(f"live extraction failed ({type(e).__name__}) — falling back to {CACHE.name}")
        print(f"  cause: {e}")
        extracted = _load_cache()
elif CACHE.exists():
    extracted = _load_cache()
    print(f"no API key — loaded cached extraction from {CACHE.name}")
else:
    raise RuntimeError(
        "No ANTHROPIC_API_KEY and no cached extraction at "
        f"{CACHE}.\nRun this cell once with a working key; every later run reads the cache."
    )

got = sum(1 for f in extracted.fields.values() if f.value is not None)

section(f"STRUCTURED — {extracted.extractor}")
print(f"  fields recovered      {got} of {len(extracted.fields)}")
print(f"  statements classified {len(extracted.statements)}")
print(f"\n  against the floor:    {floor_got} -> {got} fields, "
      f"{len(floor.statements)} -> {len(extracted.statements)} statements")

14:00:00 | INFO    | httpx                              | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 400 Bad Request"


RuntimeError: Extraction failed and no cached copy exists at reference_case_priya.json.
  cause: BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011Ce8kqiHUFmojS3ptybtRu'}
If this is a billing or auth error the key is present but unusable — fix the account, do not re-run the cell.

In [ ]:
# Every field with its provenance. `source` is the column that matters for a
# suitability file: STATED is the client's own assertion, INFERRED is derived
# from something they said, DEFAULT is a population value nobody discussed, and
# UNKNOWN is a refusal carrying the question to ask instead.
section("EXTRACTED FIELDS — value, source, confidence, evidence")

rows = []
for name, f in extracted.fields.items():
    quote = (f.evidence_quote or "").strip().replace("\n", " ")
    rows.append({
        "field":      name,
        "value":      str(f.value)[:34] if f.value is not None else "—",
        "source":     f.source.value,
        "conf":       f"{f.confidence:.2f}",
        "line":       f.evidence_line if f.evidence_line is not None else "—",
        "evidence":   (quote[:52] + "…") if len(quote) > 52 else (quote or "—"),
    })

print(pd.DataFrame(rows).to_string(index=False))

unknowns = [n for n, f in extracted.fields.items() if f.source.value == "unknown"]
if unknowns:
    print(f"\n  Refused to guess ({len(unknowns)}):")
    for n in unknowns:
        print(f"     {n}  ->  ASK: {extracted.fields[n].follow_up_question}")

## 4 · What the contract will not let a model do

Hallucination detection in this project is mostly not a check that runs afterwards. The
first layer is that a fabricated field **cannot be constructed**: `ExtractedField` refuses
to instantiate if a `STATED` value has no quote behind it, or if an `UNKNOWN` field carries
a value, a confidence above zero, or no follow-up question.

That is worth demonstrating rather than describing, so the cell below tries to build both
and catches the `ValidationError`. An extractor cannot return either shape — not because
it is well-behaved, but because the object does not exist.

In [ ]:
from pydantic import ValidationError
from contracts import ExtractedField, FactSource

section("THE PROVENANCE CONTRACT — three fabrications, three refusals")

attempts = [
    ("STATED with no evidence quote",
     dict(name="annual_salary", value=310000.0, source=FactSource.STATED, confidence=0.9)),
    ("UNKNOWN carrying a guessed value",
     dict(name="has_pension", value=False, source=FactSource.UNKNOWN, confidence=0.0,
          follow_up_question="Do you have a pension?")),
    ("UNKNOWN reporting non-zero confidence",
     dict(name="has_pension", value=None, source=FactSource.UNKNOWN, confidence=0.7,
          follow_up_question="Do you have a pension?")),
]

for label, kwargs in attempts:
    try:
        ExtractedField(**kwargs)
        print(f"  [CONSTRUCTED] {label}   <-- the contract failed to catch this")
    except ValidationError as e:
        reason = e.errors()[0]["msg"].replace("Value error, ", "")
        print(f"  [REFUSED]     {label}")
        print(f"                {reason}\n")

# And the honest refusal from the real call, which is the shape that IS allowed.
for name, f in extracted.fields.items():
    if f.source == FactSource.UNKNOWN:
        print(f"  [VALID]       {name} = None, confidence {f.confidence}, "
              f"follow-up present")
        print(f"                ASK: {f.follow_up_question}")

## 5 · Classification — prose into constraints, exposures and contradictions

Field extraction is where the rule-based floor ties the model. Classification is where the
gap is total rather than incremental: the floor scores 0, because it does not implement it.

Five kinds, and `destinations` is a **list** on purpose. A `RISK_FACT` is the category that
makes the exercise worthwhile — it arrives sounding like a preference and is really an
exposure, so one passage about ARVX is a concentration limit, a human-capital beta input
*and* a forced sector underweight at once. A statement classified and then routed nowhere
is rejected by `ClientStatement._check_routable`; naming a problem without addressing it is
the failure the taxonomy exists to prevent.

In [ ]:
from collections import Counter
from contracts import PipelineDestination, StatementKind

section(f"CLASSIFIED STATEMENTS — {len(extracted.statements)} total")

by_kind = Counter(s.kind.value for s in extracted.statements)
print(pd.DataFrame(
    [{"kind": k, "count": c} for k, c in by_kind.most_common()]
).to_string(index=False))

by_dest = Counter(d.value for s in extracted.statements for d in s.destinations)
print("\n  Routed to:")
for d, c in by_dest.most_common():
    print(f"     {d:24s} {c}")

In [ ]:
# The multi-destination case, in full. One passage, several obligations.
section("RISK_FACT — where one sentence carries three consequences")

for s in extracted.statements_of(StatementKind.RISK_FACT):
    print(f"\n  “{s.quote.strip()[:150]}”")
    print(f"     subject     {s.subject}")
    print(f"     reading     {s.summary}")
    print(f"     routes to   {', '.join(d.value for d in s.destinations)}")
    print(f"     source      {s.source.value} · confidence {s.confidence:.2f}")

section("CHALLENGE — statements that need a human, not an optimiser")
challenges = extracted.statements_of(StatementKind.CHALLENGE)
if not challenges:
    print("  none raised")
for s in challenges:
    print(f"\n  {s.summary}")
    print(f"     evidence: “{s.quote.strip()[:130]}”")

## 6 · Bridge — conversation to a validated `ProfileAgentOutput`

`build_profile_from_intake()` turns the extraction into a persona dict and runs it through
the **same** `build_profile()` the BLS personas use. The extractor supplies facts; the
formulas supply figures. Nothing in this step computes a number from prose.

Three behaviours to watch in the output:

- **Which source wins.** Facts about her circumstances — age, salary, balances, holdings —
  the client wins; no table knows her account balance. Model parameters — beta, correlation,
  sigma — the measured calibration wins. In this call the *advisor* proposed a beta of 1.2
  and she assented, hedging that it might be low. Soft assent to someone else's estimate is
  not evidence, and letting it overwrite a calibrated parameter would launder an opinion
  into a number the optimiser treats as fact. The disagreement is recorded, not discarded.
- **Every default is enumerable.** A suitability file has to distinguish a constraint the
  client asserted from one the system chose for them.
- **Required fields cannot be defaulted.** If age, salary, financial capital or income
  stability is missing, the bridge declines to build and returns the questions instead —
  the extractor's refusal carried one layer down rather than quietly reversed.

In [ ]:
from agents.profile.intake_bridge import build_profile_from_intake
from agents.profile.profile_agent import _get_discount_rate

discount_rate = _get_discount_rate(FRED_KEY)
print(f"discount rate (FRED DGS10): {discount_rate:.4f}")

result = build_profile_from_intake(extracted, discount_rate)

section("BRIDGE RESULT")
print(f"  {result.summary()}")
print(f"\n  stated    ({len(result.stated_fields)}): {result.stated_fields}")
print(f"  defaulted ({len(result.defaulted_fields)}): {result.defaulted_fields}")

if result.conflicts:
    print("\n  Source conflicts — recorded, not acted on:")
    for c in result.conflicts:
        print(f"     {c}")

if not result.built:
    print("\n  PROFILE NOT BUILT — required fields never discussed:")
    for q in result.open_questions:
        print(f"     ASK: {q}")
    raise RuntimeError("Cannot continue without a profile — see the questions above.")

profile = result.profile

In [ ]:
# The decomposition the transcript asks for.
p = profile

section(f"HUMAN-CAPITAL DECOMPOSITION — {p.client_id}")
print(f"  age                       {p.age}")
print(f"  financial capital         {p.financial_capital:>14,.0f}")
print(f"  human capital (PV)        {p.human_capital_valuation:>14,.0f}")
print(f"  total wealth              {p.total_wealth:>14,.0f}   "
      f"({p.human_capital_pct_of_total:.1f}% human capital)")
print(f"  income sigma              {p.income_volatility_sigma:>14.3f}")
print(f"  income-equity correlation {p.income_equity_correlation:>14.3f}")
print(f"  income-equity beta        {p.income_equity_beta:>14.3f}   "
      f"-> {p.human_capital_type.value}")
print(f"  implicit equity exposure  {p.implicit_equity_exposure:>14.3f}")
print(f"  effective risk budget     {p.effective_risk_budget:>14.3f}")
print(f"  PORTFOLIO EQUITY TARGET   {(p.portfolio_equity_target or 0):>14.3f}")
print(f"  employer sector           {p.industry_exposure_sector}")
print(f"  risk tolerance            {p.risk_tolerance_level.value}   (as she described it)")
print(f"  llm_role                  {p.llm_role.value}   (a model authored the inputs)")

print("\n  Current holdings — read from the transcript, not a neutral default book:")
for ticker, w in sorted(p.current_holdings.items(), key=lambda kv: -kv[1]):
    flag = "   <-- above the 10% single-name limit" if w >= 0.10 else ""
    print(f"     {ticker:12s} {w:7.1%}{flag}")

if (p.portfolio_equity_target or 0) <= 0:
    print(
        f"\n  Her career alone carries implicit equity exposure of "
        f"{p.implicit_equity_exposure:.2f} against a risk budget of "
        f"{p.effective_risk_budget:.2f}. Residual capacity for portfolio equity is "
        f"{p.portfolio_equity_target:.2f} — negative. Before she opens a brokerage "
        f"account she is already over-exposed to equities, and the employer stock "
        f"sits on top of that."
    )

In [ ]:
# Statements the Profile Agent can act on itself, versus those it carries for
# agents that own the decision. Routing is not decoration — §10 reads it back.
r = result.routed

section("ROUTED STATEMENTS")
for label, items in [
    ("concentrations (measured against her actual book)", r.concentrations),
    ("sector underweights",                              r.sector_underweights),
    ("universe exclusions -> Compliance Job 3.2",        r.exclusions),
    ("out of scope",                                     r.out_of_scope),
    ("needs a human (advisor review)",                   r.for_advisor_review),
]:
    print(f"\n  {label}:")
    if not items:
        print("     —")
    for it in items:
        print(f"     - {it}")

print(f"\n  soft views -> Black-Litterman (Allocation's input, not ours): "
      f"{len(r.soft_views)}")

# The line that keeps the mandate alive downstream. Without it the orchestrator
# builds ComplianceInput from ProfileAgentOutput, Job 3.2 receives an empty
# statement list, and every exclusion check reports as passing — vacuously.
print(f"\n  client_statements carried onto the profile: {len(profile.client_statements)}")

## 7 · Research Agent — the macro regime

Unchanged from `pipeline_demo.ipynb`: 13 FRED series, PELT structural breaks → K-means
segment check → XGBoost month-by-month labelling → 6-month rolling majority vote. The
regime is client-independent, so this is the one section of the notebook Priya's transcript
does not touch.

In [ ]:
from agents.research.research_agent import run_research_agent

banner("RESEARCH AGENT")
macro = run_research_agent(save=False, validate_crsp=False, compare_models=False)

section("MACRO REGIME SNAPSHOT")
print(f"  regime               {macro.regime_label}")
print(f"  confidence           {macro.regime_confidence:.1%}"
      f"{'   <-- low confidence, see pipeline warnings' if macro.is_low_confidence else ''}")
print(f"  prior regime         {macro.prior_regime}")
print(f"  regime change        {macro.regime_change_detected}")
print(f"  regime volatility    {macro.regime_volatility:.4f}")

## 8 · Both feedback loops — `run_pipeline`

The control plane. It makes no quantitative and no LLM decision of its own; it sequences
the agents, bounds the loops, and records what happened.

```
apply_regime_tilt  ->  Allocation <-> Risk  (Loop A, FLAG, <=3)
                              |
                              v
                       Compliance          (Loop B, FAIL, <=2)
                              |
                              v
                       AdvisorPackage
```

Watch the logs. Loop A re-enters whenever Risk returns `FLAG`, carrying the violated
constraints forward as tightened bounds — a re-proposed sleeve cannot land back on the
boundary it just breached. Loop B re-runs only what `agent_feedback` names as responsible.

In [ ]:
from agents.orchestrator.orchestrator_agent import run_pipeline

banner("ORCHESTRATED PIPELINE — run_pipeline()")
package = run_pipeline(profile, macro, fred_api_key=FRED_KEY)
m = package.metadata

In [ ]:
section("LOOP RESOLUTION")

print(f"  Loop A — Allocation <-> Risk")
print(f"     final decision      {m.final_risk_decision.value}")
print(f"     FLAG revisions      {m.risk_revisions} of 3 max"
      f"{'   (passed first try)' if m.risk_revisions == 0 else ''}")

print(f"\n  Loop B — Compliance <-> Allocation")
print(f"     final status        {m.final_compliance_status.value}")
print(f"     clearance           {package.compliance.clearance}")
print(f"     revisions           {m.compliance_revisions} of 2 max"
      f"{'   (passed first try)' if m.compliance_revisions == 0 else ''}")

if package.compliance.agent_feedback:
    print("\n  Compliance routing table (agent_feedback):")
    for agent, items in package.compliance.agent_feedback.items():
        print(f"     -> {agent}  ({len(items)} item(s))")
        for it in items:
            print(f"          {it.check}: {it.action_required[:74]}")
else:
    print("\n  No agent_feedback — nothing was routed back.")

if m.pipeline_warnings:
    print("\n  Pipeline warnings:")
    for w in m.pipeline_warnings:
        print(f"     - {w}")

## 9 · The `AdvisorPackage`

`{profile, macro, allocation, risk, compliance, metadata}`, validated end to end.

Read `proposed_portfolio` carefully: it is the **risky sleeve** and sums to 1.0 within
itself. What she actually holds is `risky_weight × sleeve + safe_weight × safe asset`, and
`risky_weight` is the deterministic Merton/BMS solve capped at her equity target — the one
number the language model is never asked to set. When that target is negative the sleeve
below describes a book that is not funded at all.

In [ ]:
from agents.allocation.adapters import ETF_SECTORS

MIN_W = 0.001    # hide dust positions below 0.1%
pkg   = package

banner("FULL ADVISOR PACKAGE")

section("SUMMARY")
print(f"  Client       {pkg.profile.client_id} · {pkg.profile.career_type} · age {pkg.profile.age}")
print(f"  Human cap.   {pkg.profile.human_capital_type.value} "
      f"(beta {pkg.profile.income_equity_beta:.2f}, "
      f"{pkg.profile.human_capital_pct_of_total:.0f}% of total wealth)")
print(f"  Eq. target   {(pkg.profile.portfolio_equity_target or 0):.3f}")
print(f"  Regime       {pkg.macro.regime_label} ({pkg.macro.regime_confidence:.0%} confidence)")
print(f"  Risk         {pkg.metadata.final_risk_decision.value} "
      f"({pkg.metadata.risk_revisions} revision(s))")
print(f"  Compliance   {pkg.metadata.final_compliance_status.value} "
      f"({pkg.metadata.compliance_revisions} revision(s)) · clearance={pkg.compliance.clearance}")

port   = pkg.allocation.proposed_portfolio
funded = sorted(((t, w) for t, w in port.items() if w >= MIN_W), key=lambda kv: -kv[1])

section(f"RISKY SLEEVE — {len(funded)} funded of {len(port)} instruments")
print(pd.DataFrame([
    {"Ticker": t, "Sector": ETF_SECTORS.get(t, "-"), "Weight": f"{w:.2%}"}
    for t, w in funded
]).to_string(index=False))

sector_w = {}
for t, w in funded:
    sec = ETF_SECTORS.get(t, "-")
    sector_w[sec] = sector_w.get(sec, 0) + w

print(f"\n  By sector (20% cap; 10% on her employer's sector "
      f"— {pkg.profile.industry_exposure_sector}):")
for s, w in sorted(sector_w.items(), key=lambda kv: -kv[1]):
    mark = "  <-- employer sector, tighter cap" if s == pkg.profile.industry_exposure_sector else ""
    print(f"     {s:24s} {w:6.2%}{mark}")

print("\n  Her employer's own stock is capped at 0% and is not in the approved")
print("  universe, so the binding constraint here is the sector cap, not the name.")

In [ ]:
section("PER-POSITION RATIONALE")
# Check 2.6 requires these to be distinct. One narrative copied across every
# ticker passes 2.2 individually and still violates Reg BI's per-recommendation
# care obligation — see §10.
for t, w in funded:
    text = pkg.allocation.allocation_rationale.get(t, "(none)")
    print(f"\n  {t}  {w:.2%}")
    print(f"     {text}")

In [ ]:
section("RISK — regime stress tests")
print(pd.DataFrame([
    {"Regime":        r,
     "Benchmark DD":  f"{ev.benchmark_drawdown:.1%}",
     "Floor":         f"{ev.drawdown_floor:.1%}",
     "Portfolio DD":  f"{ev.portfolio_drawdown:.1%}",
     "Passed":        ev.passed}
    for r, ev in pkg.risk.regime_evaluation.items()
]).to_string(index=False))

print(f"\n  annualised portfolio volatility  {pkg.risk.portfolio_volatility_annual:.2%}")
print(f"  risk decision                    {pkg.risk.risk_decision.value}")
if pkg.risk.violations:
    print("\n  Violations:")
    for v in pkg.risk.violations:
        print(f"     - {v}")

## 10 · Compliance — including the check that only fires because a human talked

Three jobs, thirteen check functions, one severity ladder: any `HIGH` → `FAIL` and no
clearance; `MEDIUM` or `LOW` → `PASS_WITH_WARNINGS`; none → `PASS`. Every decision is
deterministic, and no language model sets a pass or a fail anywhere in this agent.

Two checks are worth reading closely on this client specifically:

- **2.3a — RSU acknowledgment.** Her employer-stock concentration is well over the 10%
  threshold, so at least one position rationale must reference employer or single-name
  concentration. A portfolio that ignores the largest position in her book is not a
  fiduciary recommendation, and this is `HIGH`.
- **3.2 — client-mandate consistency.** This is the check that has been passing vacuously
  for every BLS persona in the project, because they carry no statements. Here it has her
  actual hard constraints to check the portfolio against.

In [ ]:
comp = pkg.compliance
section(f"COMPLIANCE — {comp.compliance_status.value} "
        f"(overall severity {comp.overall_severity.value})")

jobs = [("Job 1 — Risk-output audit",              "check_1"),
        ("Job 2 — Fiduciary content",              "check_2"),
        ("Job 3 — Robo-adviser (SEC IM 2017-02)",  "check_3")]

print(pd.DataFrame([
    {"Job": label,
     "Passed":     sum(c.startswith(pfx) for c in comp.passed_checks),
     "Violations": sum(v.check.startswith(pfx) for v in comp.violations),
     "Result": "PASS" if not any(v.check.startswith(pfx) for v in comp.violations) else "REVIEW"}
    for label, pfx in jobs
]).to_string(index=False))

if comp.violations:
    print("\n  Violations (severity ladder: any HIGH -> FAIL + no clearance):")
    for v in comp.violations:
        print(f"     [{v.severity.value:6s}] {v.check}  ->  {v.responsible_agent}")
        print(f"              {v.description}")
        print(f"              rule: {v.rule_reference}")
        print(f"              action: {v.action_required}\n")

print(f"\n  RECOMMENDATION: {comp.recommendation}")

In [ ]:
# Job 3.2, examined directly. The point is not that it passes — it is that it
# had something to check. Re-run the check on a copy with the statements
# stripped to show what a vacuous pass looks like in the same report.
from agents.compliance.job3_robo_adviser_checks import (
    _active_exclusions, _deterministic_findings, check_client_mandate,
)
from agents.orchestrator.orchestrator_agent import assemble_compliance_input

ci = assemble_compliance_input(profile, macro, pkg.allocation)

section("JOB 3.2 — CLIENT MANDATE, WITH AND WITHOUT HER WORDS")

active = _active_exclusions(ci.client_statements)
print(f"  statements reaching Compliance   {len(ci.client_statements)}")
print(f"  active hard-constraint exclusions {len(active)}")
for s in active:
    print(f"     - excludes '{s.subject}'  “{s.quote.strip()[:80]}”")

findings = _deterministic_findings(ci.client_statements, ci.proposed_portfolio)
print(f"\n  conflicts found in the proposed portfolio: {len(findings)}")
for f in findings:
    print(f"     [{f.directness:8s}] {f.ticker}: {f.reason}")

v_real, p_real = check_client_mandate(ci)

stripped = ci.model_copy(update={"client_statements": []})
v_vac, p_vac = check_client_mandate(stripped)

print(f"\n  with her statements:    {len(v_real)} violation(s), passed={p_real}")
print(f"  with them stripped:     {len(v_vac)} violation(s), passed={p_vac}")

print()
if v_real:
    print("  The two runs disagree, which is the whole argument: her words turned a")
    print("  clean report into a finding. Strip them and the same check reports the")
    print("  same portfolio as compliant.")
elif active:
    print("  Both report passed, but only one of them checked anything. She stated")
    print("  exclusions, the portfolio honours them, and that is a real pass — the")
    print("  stripped run reaches the same verdict without looking.")
else:
    print("  Both report passed, and neither checked anything: this client stated no")
    print("  hard exclusions, so 3.2 passes VACUOUSLY even with her statements present.")
    print("  That is not a defect — it is the honest result, and the point is that the")
    print("  count above lets a reviewer tell a vacuous pass from a real one. In the")
    print("  report itself the two are indistinguishable.")

print("\n  Either way this is why the statements have to survive every hop from the")
print("  transcript to ComplianceInput, and why the bridge assigns them onto the")
print("  profile rather than leaving them on the BridgeResult.")

In [ ]:
# Raw validated package — the full audit artefact.
section("RAW PACKAGE")
print(f"  type              {type(pkg).__name__}")
print(f"  sections          {list(pkg.model_dump().keys())}")
print(f"  llm_role recorded {pkg.profile.llm_role.value}")

SAVE_PACKAGE = False   # set True to write the run to data/outputs/
if SAVE_PACKAGE:
    out = Path(PROJECT_ROOT) / "data" / "outputs" / "advisor_package_priya.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(pkg.model_dump_json(indent=2), encoding="utf-8")
    print(f"\n  saved -> {out}")

## Notes

**What this run adds over `pipeline_demo.ipynb`.** The BLS path proves the human-capital
thesis produces differentiated portfolios across nine occupations. This path proves the
intake layer can turn an unstructured conversation into an input that same pipeline
accepts — and that the parts of the conversation which are *not* numbers (her exclusions,
her contradictions, the things she never mentioned) survive the trip instead of being
dropped at the first typed boundary.

**Where the language model is, and is not.** It reads the transcript and classifies the
statements — that is the whole of its authority here. Every figure in §6 is computed by
`build_profile()` from the extracted inputs, the same function the BLS personas use. The
profile records this as `llm_role`, so a reader of the output can tell that a model
authored its inputs without knowing which extractor was passed in.

**The beta conflict is the case worth defending out loud.** The advisor proposed 1.2 and
she assented, hedging it might be low. The calibration derives a different figure from her
income stability tier. The calibration is used, because soft assent to someone else's
estimate is not a measurement — and the disagreement is written to `conflicts` so a
reviewer can see she was told one number while the model used another. Neither silently
winning is the point.

**What was measured on the last full run** (see `agents/profile/profile_design.md`), for
comparison against your own output above. Read the field counts with one caveat: that run
was recorded against a **14-field** extraction list, and the list now has 15 — so compare
the ratio and the refusal, not the raw numerator.

| | |
|---|---|
| rule-based floor | 1 of 14 fields, 0 statements |
| structured extractor | 13 of 14 fields, 18 statements |
| the one refusal | `has_pension` — genuinely never discussed |
| human capital | $4,834,483 against $905,000 financial |
| career share of total wealth | 84.2% |
| income beta | 1.910 (equity-like) |
| implicit equity exposure | 1.608 against a 0.66 risk budget |
| portfolio equity target | **−0.945** |
| employer stock in her book | 39.8% |
| routed to advisor review | 2 challenges |

**Known limits of this notebook.**

- §3 caches the extraction to `data/outputs/reference_case_priya.json`. Delete it to force
  a fresh call; the cache is not invalidated by edits to the transcript.
- A negative equity target sends `risky_weight` to zero, so §9's sleeve describes a book
  that is not funded. Compliance Check 2.5 then reports volatility below the suitability
  band for her stated risk tolerance. That warning is the correct output — a client whose
  stated tolerance contradicts her total-wealth position is precisely the conversation a
  fiduciary is required to have, and the system's job is to refuse to let it pass
  unremarked, not to resolve it.
- `sensitivity.py` measures how far an extraction error travels: at ±20% input error,
  income volatility σ moves the mixed persona's equity share 13.5pp and the equity-like
  persona's not at all. For this client, extraction precision on σ is *not* the binding
  constraint — the corner solution is.